In [100]:
from pymongo import MongoClient
from mrjob.job import MRJob
import csv

In [101]:

client = MongoClient("mongodb://localhost:27017/")
client.drop_database("Assignment_1")
db = client["Assignment_1_sample"]
fifa_players = db["FIFA_players_sample"]

file_path = "../FIFA2026_Player/data.csv"

with open(file_path, newline="", encoding="utf-8-sig") as file:
    # DictReader reads every file as a string
    reader = csv.DictReader(file)
    records = [dict(row) for row in reader]

fifa_players.delete_many({})
if records:
    fifa_players.insert_many(records)


print(f"Inserted {len(records)} records into '{fifa_players.name}' ")

Inserted 13536 records into 'FIFA_players_sample' 


In [102]:
fifa_players.count_documents({})

13536

### Cleaning : String standardization

In [103]:
import re

doc_set = []
edits_arr = []
# listing out field names
for row in fifa_players.find():
    doc_set.append(row)

STRING_FIELDS_LIST = ["player_name", "nationality", "club", "position", "competition", "season"]

changed_count = 0
for document in doc_set:
    for field in STRING_FIELDS_LIST:
        field_value = document.get(field)
        if isinstance(field_value, str):
            edited_value = re.sub(r"\s+", " ", field_value.strip())
            document[field] = edited_value
            if edited_value != field_value:
                changed_count += 1
                edits_arr.append({
                    "val_before" : field_value,
                    "value_after" : edited_value
                })
                
print(changed_count)

# printing to see if edits have been made 
for element in edits_arr:
    print(element)
    

96
{'val_before': 'Mason Mount  ', 'value_after': 'Mason Mount'}
{'val_before': 'Real Madrid ', 'value_after': 'Real Madrid'}
{'val_before': ' Real Madrid', 'value_after': 'Real Madrid'}
{'val_before': 'Manchester  City', 'value_after': 'Manchester City'}
{'val_before': '  Toni Kroos', 'value_after': 'Toni Kroos'}
{'val_before': 'Diogo Dalot  ', 'value_after': 'Diogo Dalot'}
{'val_before': '  Dean Huijsen', 'value_after': 'Dean Huijsen'}
{'val_before': 'Barcelona ', 'value_after': 'Barcelona'}
{'val_before': ' world cup Qualifiers ', 'value_after': 'world cup Qualifiers'}
{'val_before': ' Chelsea', 'value_after': 'Chelsea'}
{'val_before': 'Nicolo  Barella', 'value_after': 'Nicolo Barella'}
{'val_before': '  Luka Modric', 'value_after': 'Luka Modric'}
{'val_before': ' league ', 'value_after': 'league'}
{'val_before': 'Yves Bissouma  ', 'value_after': 'Yves Bissouma'}
{'val_before': 'Matheus  Cunha', 'value_after': 'Matheus Cunha'}
{'val_before': ' league ', 'value_after': 'league'}
{'va

### Competition standardization
## Standardise all competition names to one of the following five values:
- "FIFA World Cup 2026"
- "Champions League" 
- "Domestic Cup" 
- "World Cup Qualifiers"
- "League"

In [104]:
# checking if the values for competition field are different from the ones allowed
standard_competition_array = ["FIFA World Cup 2026", "Champions League", "Domestic Cup", "World Cup Qualifiers","League"]

diff_array = []
for document in doc_set:
    if not(document.get('competition') in standard_competition_array ):
        diff_array.append(document.get("competition"))

print(len(diff_array))

long_str = ""
for name in diff_array:
    long_str = long_str  + name
print(long_str)

79
leagueleagueLEAGUELEAGUECHAMPIONS LEAGUEworld cup QualifiersCHAMPIONS LEAGUEleagueleagueleagueCHAMPIONS LEAGUECHAMPIONS LEAGUECHAMPIONS LEAGUEChampions leagueWorld Cup qualifiersleagueleaguechampions LeagueChampions leagueleagueleagueWORLD CUP QUALIFIERSWorld Cup qualifiersCHAMPIONS LEAGUEFIFA WORLD CUP 2026DOMESTIC CUPleagueLEAGUEWORLD CUP QUALIFIERSCHAMPIONS LEAGUEleagueleagueLEAGUEleagueChampions leaguedomestic cupleagueFIFA world cup 2026World Cup qualifiersleagueFIFA world cup 2026LEAGUEDomestic cupLEAGUEFIFA world cup 2026fifa World Cup 2026leagueCHAMPIONS LEAGUEleagueleagueleagueWorld Cup qualifiersleaguedomestic cupleagueleagueLEAGUEleagueleagueLEAGUEleaguedomestic cup


In [105]:
count = 0
new_Arr = []

# This function takes in the string, splitted using the delimiter " " and checks if the first word is "fifa". If that is the case then it converts it to FIFA, otherwise a simple first letter capitalization

# The strings are then joined usign the join function
def capitalize_first(array):
    if array[0].lower() == "fifa":
        array = [word.upper() if word.lower == "fifa" else word.capitalize() for word in array]
        print(array)
    else:
        array = [word.capitalize() for word in array]
    joined_word = " ".join(array)
    return joined_word

for document in doc_set:
    if document.get("competition") not in standard_competition_array:
        splitted_words_array = document.get("competition").split(" ")
        first_letter_capitalized_word = capitalize_first(splitted_words_array)
        document["competition"] = first_letter_capitalized_word
        
# This function counts the number of strings not a part of standardised strings
for document in doc_set:
    if document.get("competition") not in standard_competition_array:
        count += 1
        new_Arr.append(document["competition"])

print(new_Arr)  


['Fifa', 'World', 'Cup', '2026']
['Fifa', 'World', 'Cup', '2026']
['Fifa', 'World', 'Cup', '2026']
['Fifa', 'World', 'Cup', '2026']
['Fifa', 'World', 'Cup', '2026']
['', '', '', '', '', '', '', '', '', '', 'Fifa World Cup 2026', '', '', 'Fifa World Cup 2026', 'Fifa World Cup 2026', '', '', '', '', 'Fifa World Cup 2026', 'Fifa World Cup 2026', '']


In [106]:
" ".join(["ape"])

'ape'

In [107]:
for doc in doc_set:
    if doc.get("competition") not in standard_competition_array:
        print(doc["competition"])











Fifa World Cup 2026


Fifa World Cup 2026
Fifa World Cup 2026




Fifa World Cup 2026
Fifa World Cup 2026



In [108]:
for doc in doc_set:
    if(doc.get("competition") == "Champions Leage"):
        print(doc)

### Missing identifiers: After string standardisation, remove any record with a missing value in one or more required identifier fields: player_name, competition, match_id

In [ ]:
# checking length of documents before removal
print(len(doc_set)) # 13536

13536


In [122]:
IDENTIFIER_FIELDS = ["player_name", "competition", "match_id"]
cleaned_records = []
missing_count = 0


def is_missing(value):
    return value is None or (isinstance(value, str) and value.strip() == "")


def remove_missing_value_for_identifirs(documents):
    global missing_count
    for document in documents:
        document_is_missing = any(is_missing(document.get(field)) for field in IDENTIFIER_FIELDS)
        if document_is_missing:
            missing_count += 1
        else:
            cleaned_records.append(document)


remove_missing_value_for_identifirs(doc_set)
print(missing_count)
print(len(doc_set) - missing_count == len(cleaned_records))

print(len(cleaned_records))

43
True
13493


### Invalid numeric values: 
- Remove any record containing a negative value in any of the following fields:
    - market_value_eur_m, minutes_played, goals, assists, shots_on_target, tackles, match_rating, or fouls_committed

In [128]:
INVALID_NUMERIC_VALUE_FIELDS = ["market_value_eur_m", "minutes_played", "goals", "assists", "shots_on_target", "tackles", "match_rating", "fouls_committed" ]

records_with_negatives_count = 0
invalid_value_dictionary = []
cleaned_records_after_removing_invlaid_numeric = []
for document in cleaned_records:
    has_negative = False
    for field in INVALID_NUMERIC_VALUE_FIELDS:
        if(document.get(field) == ""):
            continue
        if(float(document.get(field)) < 0):
            has_negative = True
            invalid_value_dictionary.append({field: document.get(field)})
    if has_negative:
        records_with_negatives_count += 1
    else:
        cleaned_records_after_removing_invlaid_numeric.append(document)
        
print(records_with_negatives_count)
print(invalid_value_dictionary)
print(len(cleaned_records_after_removing_invlaid_numeric))

96
[{'market_value_eur_m': '-2.72'}, {'fouls_committed': '-3'}, {'goals': '-2'}, {'assists': '-5'}, {'assists': '-3'}, {'tackles': '-3'}, {'tackles': '-1'}, {'goals': '-4'}, {'assists': '-4'}, {'tackles': '-5'}, {'shots_on_target': '-2'}, {'minutes_played': '-7'}, {'tackles': '-5'}, {'shots_on_target': '-3'}, {'assists': '-4'}, {'match_rating': '-5.1'}, {'market_value_eur_m': '-1.67'}, {'minutes_played': '-27'}, {'assists': '-3'}, {'minutes_played': '-12'}, {'goals': '-2'}, {'shots_on_target': '-5'}, {'goals': '-3'}, {'match_rating': '-1.43'}, {'fouls_committed': '-3'}, {'tackles': '-4'}, {'shots_on_target': '-4'}, {'assists': '-1'}, {'market_value_eur_m': '-3.21'}, {'market_value_eur_m': '-7.16'}, {'shots_on_target': '-1'}, {'fouls_committed': '-5'}, {'assists': '-4'}, {'tackles': '-4'}, {'assists': '-5'}, {'assists': '-2'}, {'tackles': '-5'}, {'market_value_eur_m': '-7.2'}, {'market_value_eur_m': '-2.82'}, {'goals': '-2'}, {'market_value_eur_m': '-1.86'}, {'shots_on_target': '-5'}, {

In [129]:
print(len(cleaned_records))
print(len(cleaned_records_after_removing_invlaid_numeric))

13493
13397


### Unknown nationality: For each record where nationality = "Unknown", use records with the same

player_name, age, and club to infer the nationality. If the known nationalities in all matching records
are the same, replace "Unknown" with that nationality; otherwise, retain "Unknown".

#### Checking unknown nationality counts before cleaning

In [134]:
CHECK_FIELD = "nationality"

empArr = []
unknown_nationality_count_before_cleaning = 0
for document in doc_set:
    if document.get("nationality") == "Unknown":
        unknown_nationality_count_before_cleaning += 1
        empArr.append({"nationality":  document.get("nationality")})
    

print(len(empArr))

219


#### Nationality counts missing value filling

In [142]:
# creating an empty object
nationality_groups = {}

for document in cleaned_records_after_removing_invlaid_numeric:
    nationality = document.get("nationality")
    # checking if there exists a nationality with a value not equal to 'Unknown'
    if nationality and nationality != "Unknown":
        key = (document.get("player_name"), document.get("age"), document.get("club"))
        if key not in nationality_groups:
            nationality_groups[key] = set()
        nationality_groups[key].add(nationality)
        
print(nationality_groups)

# fixing the unknown values 

fixed_count = 0

for document in cleaned_records_after_removing_invlaid_numeric:
    if document.get("nationality") == "Unknown":
        key = (document.get('player_name'), document.get("age"), document.get("club"))
        # in case there are no documents it returns set
        candidates = nationality_groups.get(key, set())
        if len(candidates) == 1:
            document["nationality"] = next(iter(candidates))
            fixed_count += 1

{('Cristian Romero', '32', 'Tottenham Hotspur'): {'Argentina'}, ('Santiago Gimenez', '33', 'AC Milan'): {'Mexico'}, ('Nuno Mendes', '24', 'Paris Saint-Germain'): {'Portugal'}, ('Florian Wirtz', '37', 'Liverpool'): {'Germany'}, ('Michael Olise', '35', 'Bayern Munich'): {'France'}, ('Loic Bade', '17', 'Sevilla'): {'France'}, ('Jordan Henderson', '32', 'Ajax'): {'England'}, ('Rodrygo', '21', 'Real Madrid'): {'Brazil'}, ('Karim Benzema', '24', 'Al Ittihad'): {'France'}, ('Diogo Dalot', '17', 'Manchester United'): {'Portugal'}, ('Anthony Gordon', '31', 'Newcastle United'): {'England'}, ('Erik ten Hag XI', '39', 'Manchester United'): {'Netherlands'}, ('Dusan Vlahovic', '33', 'Juventus'): {'Serbia'}, ('Victor Osimhen', '24', 'Galatasaray'): {'Nigeria'}, ('Dani Carvajal', '36', 'Real Madrid'): {'Spain'}, ('Ethan Nwaneri', '27', 'Arsenal'): {'England'}, ('Gabriel Martinelli', '24', 'Arsenal'): {'Brazil'}, ('Reece James', '29', 'Chelsea'): {'England'}, ('Neymar Jr', '32', 'Santos'): {'Brazil'}, 

In [143]:
unknown_nationality_count_after_cleaning = 0
for document in cleaned_records_after_removing_invlaid_numeric:
    if document.get("nationality") == "Unknown":
        unknown_nationality_count_after_cleaning += 1
    
print(unknown_nationality_count_after_cleaning)

0
